# 11 Predict final ranking

Predicts `final_ranking.parquet` (January, February, June and July 2026) with the final models in `model/final`,
written by `09_corrector`. Nothing is trained here; the same models as for the daily leaderboard are used.

In [1]:
TEAM = "talented-jelly"
UPLOAD = True        # True copies the file to the team bucket (needs the challenge keys)

In [2]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "model/runner.py").is_file())
sys.path.insert(0, str(ROOT / "model"))
import runner
from runner import TARGET

MODELS = ROOT / "model/final"
json.loads((MODELS / "manifest.json").read_text())

{'A': {'run_id': '2a97c58055c641429638c1f74fa74751',
  'run_name': '291_BaseA_CatBoost'},
 'B': {'run_id': 'c492e5c19f3a4e6cac86cf956a48d68a',
  'run_name': '292_BaseB_CatBoost'},
 'C': {'run_id': '0645226cdf6d4b3eadbe7dc06e028ffa',
  'run_name': '294_Classifier_CatBoost'},
 'corrector': {'run_id': 'a305f283e7b144f6aef1e085d84bf44f',
  'run_name': '295_Corrector'}}

## Prediction

Base models, classifier, corrector, mixture, rule and LOBT window (`runner.predict`).

In [3]:
frame = pd.read_parquet(ROOT / "data/03_features/final_ranking.parquet")
frame = frame.loc[frame["PHASE_mvt"].eq("DEP")].reset_index(drop=True)
frame = runner.predict(frame, MODELS)
frame["month"] = pd.to_datetime(frame["MVT_TIME_UTC_mvt"], utc=True).dt.month
print(f"{len(frame):,} departures, months {sorted(frame['month'].unique().tolist())}")
frame[["base_prediction", "base_normal_prediction", "p_artifact", "prediction"]].describe().round(2)

670,790 departures, months [1, 2, 6, 7]


,base_prediction,base_normal_prediction,p_artifact,prediction
count,670790.00,670790.00,670790.00,670790.00
mean,1014.29,999.27,0.00,1011.77
std,550.94,366.24,0.04,582.75
min,0.00,0.00,0.00,0.00
25%,757.60,755.29,0.00,751.34
50%,945.89,942.14,0.00,938.69
75%,1188.77,1179.90,0.00,1180.53
max,93329.94,4259.44,1.00,110753.19


In [4]:
frame.groupby(["month", "ADEP_mvt"])["prediction"].median().unstack(0).round(0)

month,1,2,6,7
ADEP_mvt,,,,
EDDF,905.0,839.0,869.0,862.0
EDDM,846.0,761.0,763.0,757.0
EGLL,1283.0,1287.0,1343.0,1343.0
EHAM,782.0,736.0,699.0,820.0
LEBL,832.0,865.0,949.0,940.0
LEMD,978.0,974.0,1025.0,1009.0
LFPG,926.0,899.0,926.0,915.0
LIRF,971.0,954.0,1083.0,1131.0
LSZH,770.0,713.0,697.0,705.0


## Submission file

IDs and order of the challenge template `final_submitting.parquet`: every departure of the final ranking file
exactly once, in the template's order.

In [5]:
template = pd.read_parquet(ROOT / "data/01_raw/final_submitting.parquet")
assert template["MVT_ID_mvt"].is_unique and frame["MVT_ID_mvt"].is_unique, "duplicate flight IDs"
assert set(template["MVT_ID_mvt"]) == set(frame["MVT_ID_mvt"]), "predicted departures differ from the template"
submission = template[["MVT_ID_mvt"]].copy()
submission[TARGET] = submission["MVT_ID_mvt"].map(frame.set_index("MVT_ID_mvt")["prediction"])
assert submission[TARGET].notna().all() and np.isfinite(submission[TARGET]).all(), "missing predictions"
assert submission["MVT_ID_mvt"].equals(template["MVT_ID_mvt"]) and list(submission.columns) == list(template.columns)

path = ROOT / "submissions" / f"{TEAM}_final.parquet"
path.parent.mkdir(exist_ok=True)
submission.to_parquet(path, index=False)
print(f"{path.relative_to(ROOT)}: {len(submission):,} rows, same flights and order as final_submitting.parquet")
submission[TARGET].describe().round(1)

submissions/talented-jelly_final.parquet: 670,790 rows, same flights and order as final_submitting.parquet


count    670790.0
mean       1011.8
std         582.7
min           0.0
25%         751.3
50%         938.7
75%        1180.5
max      110753.2
Name: TAXITIME_SEC_mvt, dtype: float64

In [6]:
if UPLOAD:
    from pyopensky.s3 import S3Client
    S3Client().s3client.fput_object(f"prc-2026-{TEAM}", path.name, str(path))
    print("uploaded", path.name)

uploaded talented-jelly_final.parquet
